<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 01. AlexNet — ImageNet Classification with Deep Convolutional Neural Networks

- **원 논문:** [ImageNet Classification with Deep Convolutional Neural Networks](https://proceedings.neurips.cc/paper_files/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf)
- **저자 / 발표:** Alex Krizhevsky, Ilya Sutskever, and Geoffrey E. Hinton · NeurIPS 2012 (2012)
- **난이도 / 예상 시간:** 초중급 · 약 85분
- **선수 지식:** Conv2d, ReLU, max pooling, broadcasting

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

AlexNet의 ReLU, channel-local response normalization(LRN), 겹치는 max pooling, 5-convolution/3-linear-stage 구성을 축소 모델에서 구현한다. LRN 수식의 channel window와 pooling stride를 수치·shape로 검증하고 합성 색상 패턴에서 짧게 최적화한다.

**원 논문과 다른 것**

원 논문은 ImageNet 1.2M 영상, 1000 classes, 약 60M parameters, 두 GPU로 분할된 5 convolution + 3 fully-connected network를 수일간 학습한다. 여기서는 64×64 합성 영상 24개, 최대 32 channels와 4 classes를 사용하며 data augmentation과 multi-GPU는 생략한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3.1 ReLU Nonlinearity, 식 f(x)=max(0,x) | 각 convolution 직후 `F.relu` | 모델 코드에서 다섯 conv 모두 ReLU 뒤에 배치 |
| §3.3 Local Response Normalization, Eq. (1) | `paper_lrn`의 인접 channel 제곱합 | 단일 channel spike의 중심/이웃 정규화 값을 assert |
| §3.4 Overlapping Pooling: z=3, s=2 | `max_pool2d(..., kernel_size=3, stride=2)` 비교 셀 | stride 2와 stride 3의 출력 크기 및 heatmap 비교 |
| §3.5 Overall Architecture와 Fig. 2: 5 conv + 3 FC, FC dropout | `AlexNetMini`의 conv1…conv5와 두 hidden/output linear layers | 모듈 개수, logits shape, loss 감소를 검증 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\operatorname{ReLU}(x)=\max(0,x),\qquad
p(y=k\mid x)=\operatorname{softmax}(z)_k
$$

- ReLU는 음수 activation을 0으로 만들어 깊은 CNN의 gradient 전달을 돕습니다.
- $z$는 마지막 linear layer의 logit이고 softmax는 $K$개 class 확률로 변환합니다.
- 코드에서 feature extractor, dropout, classifier를 따로 두어 AlexNet의 블록 경계를 보이게 합니다.
- 입력 $[B,1,H,W]$는 feature map을 거쳐 logit $[B,K]$로 변합니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3.1 ReLU Nonlinearity, 식 f(x)=max(0,x)
- **노트북 구현:** 각 convolution 직후 `F.relu`
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** 모델 코드에서 다섯 conv 모두 ReLU 뒤에 배치를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 원 논문은 ImageNet 1.2M 영상, 1000 classes, 약 60M parameters, 두 GPU로 분할된 5 convolution + 3 fully-connected network를 수일간 학습한다. 여기서는 64×64 합성 영상 24개, 최대 32 channels와 4 classes를 사용하며 data augmentation과 multi-GPU는 생략한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. 무엇을 재현하는가

AlexNet의 역사적 성능은 대규모 데이터·GPU 학습과 함께 나온 결과입니다. 이 실습은
그 정확도를 재현하지 않고 §3의 세 가지 구체적 선택—ReLU, Eq. (1)의 LRN,
`3×3/stride 2` overlapping pooling—을 분리해서 확인한 뒤 작은 5-conv 모델에 합칩니다.
PyTorch의 `LocalResponseNorm`은 `alpha/n` 관례를 쓰므로 논문 수식을 눈으로 그대로
추적할 수 있도록 직접 구현합니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** image [B,1,H,W] → five convolution stages → logits [B,K]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(1)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def make_color_patches(repeats=6, size=64):
    generator = torch.Generator().manual_seed(202)
    images, labels = [], []
    for index in range(4 * repeats):
        label = index % 4
        image = torch.zeros(3, size, size)
        row = 7 + (label // 2) * 30
        col = 7 + (label % 2) * 30
        image[label % 3, row : row + 18, col : col + 18] = 1.0
        image[(label + 1) % 3, row + 4 : row + 14, col + 4 : col + 14] = 0.45
        noise = 0.035 * torch.randn(image.shape, generator=generator)
        images.append((image + noise).clamp(0, 1))
        labels.append(label)
    return torch.stack(images), torch.tensor(labels)


alex_x, alex_y = make_color_patches()
alex_x, alex_y = ACCELERATOR.move(alex_x, alex_y)
assert alex_x.shape == (24, 3, 64, 64)
print(ACCELERATOR.summary())
print("synthetic ImageNet stand-in:", tuple(alex_x.shape))

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3.3 Local Response Normalization, Eq. (1)
- **이 셀의 책임:** `paper_lrn`의 인접 channel 제곱합
- **Tensor shape 계약:** image [B,1,H,W] → five convolution stages → logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 단일 channel spike의 중심/이웃 정규화 값을 assert. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: 논문 Eq. (1)을 구현하세요.
def paper_lrn(x, n=5, k=2.0, alpha=0.0001, beta=0.75):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


spike = torch.zeros(1, 7, 1, 1)
spike[:, 3] = 2.0
normalized_spike = paper_lrn(spike)


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 평가·완료 증거

- **원문 위치:** §3.4 Overlapping Pooling: z=3, s=2
- **이 셀의 책임:** `max_pool2d(..., kernel_size=3, stride=2)` 비교 셀
- **Tensor shape 계약:** image [B,1,H,W] → five convolution stages → logits [B,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** stride 2와 stride 3의 출력 크기 및 heatmap 비교. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: 7×7 grid에 3×3 max pooling을 적용하세요.
# stride=2(overlap)와 stride=3(non-overlap)의 출력 shape를 assert하고 나란히 그리세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** §3.1 ReLU Nonlinearity, 식 f(x)=max(0,x) / §3.3 Local Response Normalization, Eq. (1) / §3.4 Overlapping Pooling: z=3, s=2 / §3.5 Overall Architecture와 Fig. 2: 5 conv + 3 FC, FC dropout
- **이 셀의 책임:** 각 convolution 직후 `F.relu` / `paper_lrn`의 인접 channel 제곱합 / `max_pool2d(..., kernel_size=3, stride=2)` 비교 셀 / `AlexNetMini`의 conv1…conv5와 두 hidden/output linear layers
- **Tensor shape 계약:** image [B,1,H,W] → five convolution stages → logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 모델 코드에서 다섯 conv 모두 ReLU 뒤에 배치 / 단일 channel spike의 중심/이웃 정규화 값을 assert / stride 2와 stride 3의 출력 크기 및 heatmap 비교 / 모듈 개수, logits shape, loss 감소를 검증. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: 5개의 convolution과 논문식 LRN/overlapping pool을 갖는 축소 AlexNet을 만드세요.
class AlexNetMini(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, classes=4):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


alexnet = AlexNetMini().to(DEVICE)


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 5 · 논문 주장 확인 실험

- **원문 위치:** §3.5 Overall Architecture와 Fig. 2: 5 conv + 3 FC, FC dropout
- **이 셀의 책임:** `AlexNetMini`의 conv1…conv5와 두 hidden/output linear layers
- **Tensor shape 계약:** image [B,1,H,W] → five convolution stages → logits [B,K]
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** 모듈 개수, logits shape, loss 감소를 검증. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: logits shape, convolution 개수, parameter 수 범위를 assert하세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### 구현 단계 6 · 평가·완료 증거

- **원문 위치:** §3.5 Overall Architecture와 Fig. 2: 5 conv + 3 FC, FC dropout
- **이 셀의 책임:** `AlexNetMini`의 conv1…conv5와 두 hidden/output linear layers
- **Tensor shape 계약:** image [B,1,H,W] → five convolution stages → logits [B,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 모듈 개수, logits shape, loss 감소를 검증. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: 모델을 train mode로 두고 12 step 학습하세요.
# loss 감소를 검증하고 curve 및 최종 합성 정확도를 표시하세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.